In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import os

os.environ["CUDA_LAUNCH_BLOCKING"] = "1"

In [3]:
from pathlib import Path

import torch
import torch.multiprocessing as mp
import tqdm
from jppype import vscode_theme
from torch_geometric.loader import DataLoader
from torch_geometric.transforms import ToDevice

from fundus_vessels_toolkit.segment_to_graph.models.dataset import VBranchDigraphDataset

vscode_theme()

HTML(value="<style>\n        .cell-output-ipywidget-background {\n                background: transparent !imp…

In [4]:
PATH = [
    Path("/run/media/gaby/GREY SSD/PostDoc/DATA/Fundus/" + folder)
    for folder in ["GAVE-train", "MAPLES-DR", "Fundus-AV"]
]
RAW = [path / "1-images" for path in PATH]
AV = [path / "2-av-pred_CLEMENT" for path in PATH]
TOPO = [path / "3-topo" for path in PATH]

In [5]:
dataset = VBranchDigraphDataset.load_from_dirs(RAW, TOPO, av_dir=AV, resize_to=1024)

Found 215 branch digraphs...


Processing...
Done!
Preloading dataset: 100%|██████████| 215/215 [00:18<00:00, 11.35it/s]


In [6]:
m, digraph, _ = dataset.draw_jppype(0, augment=True, test=True)
m

[ WARN:0@0.336] global loadsave.cpp:1617 imencodeWithMetadata Unsupported depth image for selected encoder is fallbacked to CV_8U.


GridBox(children=(HTML(value='<h3 style="text-align: center;">Predicted</h3>'), HTML(value='<h3 style="text-al…

In [7]:
from fundus_vessels_toolkit.segment_to_graph.models.digraph_model import (
    BranchDigraphModel,
    BranchFeaturesEfficientNetV2S,
    Gatv2GCN,
)

model = BranchDigraphModel(BranchFeaturesEfficientNetV2S(), Gatv2GCN(n_in=784, n_out=512))
model.cuda()


BranchDigraphModel(
  (img_feature_extractor): BranchFeaturesEfficientNetV2S(
    (net): Sequential(
      (0): Conv2dNormActivation(
        (0): Conv2d(3, 24, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
        (1): BatchNorm2d(24, eps=0.001, momentum=0.1, affine=True, track_running_stats=True)
        (2): SiLU(inplace=True)
      )
      (1): Sequential(
        (0): FusedMBConv(
          (block): Sequential(
            (0): Conv2dNormActivation(
              (0): Conv2d(24, 24, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
              (1): BatchNorm2d(24, eps=0.001, momentum=0.1, affine=True, track_running_stats=True)
              (2): SiLU(inplace=True)
            )
          )
          (stochastic_depth): StochasticDepth(p=0.0, mode=row)
        )
        (1): FusedMBConv(
          (block): Sequential(
            (0): Conv2dNormActivation(
              (0): Conv2d(24, 24, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False

In [8]:
for i, data in enumerate(tqdm.tqdm(DataLoader(dataset, batch_size=3, num_workers=5))):
    data = data.cuda()
    av_p, dir_p, parent_p = model(data.cuda())
    (av_p.mean() + dir_p.mean() + parent_p.mean()).backward()

100%|██████████| 72/72 [00:20<00:00,  3.46it/s]
